# 03 · Reviewed labels, frozen split and first classifiers (Lecture 4)

**Question:** with reviewed labels and a fixed account + time holdout, how well do simple classifiers do?

Lecture 4 asked us to freeze the split before fitting anything, train on adjudicated labels only, and compare a
first fastText model against simple baselines on the same validation rows. Participation question 1: should
future memos come from seen accounts, unseen accounts or both? We chose both:

- **Test A:** later transactions from accounts seen in training (time drift)
- **Test B:** later transactions from held-out accounts (new-customer transfer)

Our PA4 result (Vanessa's notebook): 400 memos, 20 accounts (16 development, 4 held out); 50 epochs gave 65%
validation accuracy (39/60); Test A 18/30 (60%), Test B 9/30 (30%), with sparse classes and unseen merchants
as the main errors.

> Runs on DataHub. Cells show aggregate tables and charts only; files with memo text (errors, examples, predictions) stay in `outputs/` and are never printed here or committed.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
CONFIG = os.environ.get("CAPSTONE_CONFIG", "config.yaml")

import pandas as pd
from IPython.display import Image, display
from src.config import load_config
from src.main import run
OUT = load_config(CONFIG)["paths"]["out_dir"]
pd.set_option("display.max_colwidth", 80)

## Labels, class counts and leakage checks

In [ ]:
m = run(CONFIG, "prepare")
m["labels"]

In [ ]:
pd.read_csv(OUT / "class_counts.csv", index_col=0)

In [ ]:
lab = pd.read_csv(OUT / "labels_prepared.csv", parse_dates=["date"])
lab.groupby("split").agg(rows=("category", "size"), accounts=("account_key", "nunique"),
                         first_date=("date", "min"), last_date=("date", "max"))

`prepare` fails if a Test B account appears in train/validation or a transaction appears twice.
The table above shows the date windows the labels actually use.

## Baselines on the same validation rows

In [ ]:
m = run(CONFIG, "baseline")
pd.read_csv(OUT / "baseline.csv")

The majority class is the floor. TF-IDF + logistic regression is Lecture 3's reference baseline.
The two fastText rows are the Lecture 4 starter settings and our PA4 setting, rerun on the current labels, so
the gain from tuning in notebook 04 is measured against them.